# Tutorial: Palette catalog summaries

**Audience:** Python beginners who want to summarize CSV catalogs.

**Prerequisites:** Basic Python functions, dictionaries, lists, and file paths.

## Learning goals

By the end, you can read a UTF-8 CSV by header name, normalize grouping keys, collect unique labels, and sort a summary deterministically.


## Outline

1. Import standard-library tools and define the summarizer.
2. Group the palette catalog by normalized primary color.
3. Inspect the ordered result.
4. Extend the approach in the exercises.


## Step 1 â€” Define the grouping function

`csv.DictReader` uses the header row, so column order does not matter. Each primary color and product type is stripped; primary colors are also uppercased. A set keeps product types unique while every row still increments the count.


In [ ]:
from __future__ import annotations

import csv
from collections import defaultdict
from pathlib import Path


def summarize_palettes(csv_path):
    """Return palette groups ordered by descending count, then primary."""
    groups = defaultdict(lambda: {"count": 0, "product_types": set()})

    with Path(csv_path).open("r", encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle):
            primary = row["Primary (Hex)"].strip().upper()
            product_type = row["Product Type"].strip()
            groups[primary]["count"] += 1
            groups[primary]["product_types"].add(product_type)

    summary = [
        {
            "primary": primary,
            "count": details["count"],
            "product_types": sorted(details["product_types"]),
        }
        for primary, details in groups.items()
    ]
    return sorted(summary, key=lambda item: (-item["count"], item["primary"]))


## Step 2 â€” Summarize the catalog

Call the function with the supplied catalog. The result is a list of dictionaries, one per normalized primary color.


In [ ]:
palette_summary = summarize_palettes('/inputs/colors.csv')


## Step 3 â€” Inspect the result

The first few records show the most frequent colors. The assertions document the three fields and ordering expected from every record.


In [ ]:
assert all(set(item) == {"primary", "count", "product_types"} for item in palette_summary)
assert palette_summary == sorted(
    palette_summary,
    key=lambda item: (-item["count"], item["primary"]),
)
palette_summary[:5]


## Exercises

1. Make a small CSV whose primary colors differ only by case or surrounding spaces; predict its single grouped result.
2. Add a duplicate row and confirm that `count` increases while `product_types` remains unique.
3. Try a product type containing a comma (quoted by the CSV writer) and verify it remains one value.
4. Create a header-only catalog and confirm that the function returns an empty list.
